# Multiple Lineare Regression

Bisher haben wir uns nur einfache lineare Regressionsprobleme angesehen. Aber lineare Regression kann auch mit mehr als einer erklärenden Variable verwendet werden, um die Zielvariable zu beschreiben.

## Lernziele

Am Ende dieses Notebooks sollten Sie in der Lage sein
- multiple lineare Regression mit scikit-learn anzuwenden.
- multiple lineare Regressionsmodelle zu interpretieren.
- zu erklären, warum es besser ist, **adjustiertes $R^2$** statt $R^2$ zu verwenden, um multiple lineare Regressionsmodelle zu vergleichen.

## Wiederholung

### Das Multiple Regressionsmodell

Multiple lineare Regression ist der einfachen linearen Regression sehr ähnlich, außer dass die abhängige Variable $y$ durch $k$ unabhängige Variablen $x_1, \dots, x_k$ beschrieben wird

$$
y = b_0 + b_1 x_1 + b_2 x_2 + \dots + b_k x_k + e
$$

Die Formel für unser **vorhergesagtes Modell** ist gegeben durch

$$\hat{y} = b_0  + b_1  x_1 + b_2  x_2 + \dots + b_k  x_k$$

wobei $b_0$ der geschätzte Achsenabschnitt und $b_1, b_2,$... die Koeffizienten unserer Features sind.
* **Achsenabschnitt**: Die Interpretation bleibt dieselbe wie bei der einfachen linearen Regression. Es ist der Wert für $y$, wenn alle $x$ gleich 0 sind.
* **Koeffizienten**: Bezüglich der Interpretation der Koeffizienten müssen wir präziser sein: $b_i$ ist die Änderung in $y$ bei einer Änderung von $x_i$ um eine Einheit, während **alle anderen Variablen konstant gehalten werden**

## Beginnen wir mit einem Beispiel

Für dieses Beispiel werden wir wieder einen Auto-Datensatz verwenden. Diesmal wird er etwas umfangreicher mit mehr Beobachtungen und Features sein, aber unser Ziel ist immer noch, `mpg` basierend auf verschiedenen Auto-Charakteristiken vorherzusagen.

In [ ]:
# Import packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Set figure stile and size for entire notebook
sns.set_style("ticks")
plt.rcParams["figure.figsize"] = (7,4)

In [ ]:
# Import dataset 
cars = pd.read_csv("data/cars_multivariate.csv")
cars.head()

## Einfache Lineare Regression

Wie zuvor werden wir zunächst ein einfaches lineares Regressionsmodell mit dem Feature `horsepower` verwenden und sehen, wie gut es zu unseren Daten passt.

In [ ]:
# Plot relationship between horsepower and mpg 
plt.scatter(cars.horsepower, cars.mpg)
plt.ylabel('mpg')
plt.xlabel('horsepower');

In [ ]:
# Define feature and target variable
X = cars[['horsepower']]
y = cars['mpg']

X.shape

In [ ]:
# Fit linear regression model
lin_reg = LinearRegression()
lin_reg.fit(X, y)

# Calculate r-squared 
y_hat = lin_reg.predict(X)
print("R-squared:", round(r2_score(y, y_hat),3))

In [ ]:
# Plot data with regression line
plt.scatter(X, y)
plt.plot(X, y_hat, '-', color='orange', linewidth=2)
plt.ylabel('mpg')
plt.xlabel('horsepower');

Laut $R^2$ kann das Feature `horsepower` 60,6% der Varianz in unserer Zielvariablen erklären.

## Multiple Lineare Regression

Anstatt `horsepower` als einzige unabhängige Variable zur Vorhersage von `mpg` zu verwenden, möchten wir vielleicht andere unabhängige Variablen in das Modell aufnehmen, um die Anpassung zu verbessern. Versuchen wir, `weight` zum Modell hinzuzufügen.

In [ ]:
# Plot relationship between weight and mpg 
# We can use seaborns .lmplot function to plot our data including a regression line 
sns.lmplot(x='weight', y='mpg', data=cars, ci=None);

In [ ]:
# Define model input X with two features
X2 = cars[['horsepower', 'weight']]
y2 = cars.mpg

X2.shape

In [ ]:
# Fit linear regression model
lin_reg2 = LinearRegression()
lin_reg2.fit(X2, y2)

# Calculate intercept and coefficient
intercept = lin_reg2.intercept_
coefficients = lin_reg2.coef_
print("Intercept:", intercept.round(4))
print("Coefficients:", coefficients.round(4))

In [ ]:
# Calculate r-squared 
y_hat2 = lin_reg2.predict(X2)
print("R-squared:", round(r2_score(y2, y_hat2),3))

### Modellinterpretation

Beschreibt dieses Modell die Varianz von `mpg` besser als die einfache lineare Regression? Es scheint so. Dieses Modell erklärt etwa 70% der Variation in `mpg`.

Unser multiples Regressionsmodell ist gegeben durch

$$ \hat{mpg} = 45.6402 - 0.0473 \times horsepower - 0.0058 \times weight $$

Lassen Sie uns interpretieren, was wir hier sehen können...
* Das erwartete `mpg` für ein Auto mit einer `horsepower` und einem `weight` von 0 beträgt 45,6402 (theoretisch...).
* Wir würden erwarten, dass `mpg` um 0,0473 abnimmt, wenn `horsepower` um 1 steigt, **unter Konstanthaltung von `weight`**.
* Wir würden erwarten, dass `mpg` um 0,0058 abnimmt, wenn `weight` um 1 steigt, **unter Konstanthaltung von `horsepower`**.

### Vorhersagen für neue Autos

Was ist das vorhergesagte `mpg` für ein Auto mit 200 `horsepower` und einem `weight` von 3500?

$$
\hat{mpg} = 45.6402 - 0.0473 (200) - 0.0058 (3500) = 15.88
$$

Wir würden erwarten, dass das `mpg` des Autos 15,88 beträgt.
Dies kann natürlich auch mit unserem Modell berechnet werden. Wir müssen nur die `.predict()` Funktion aufrufen und ihr die Werte von X geben.

In [ ]:
# Make predictions for new car with horsepower=200 and weight=3500
new_car = pd.DataFrame({'horsepower': [200], 'weight': [3500]})
y_prediction = lin_reg2.predict(new_car)
print("Prediction for new car:", y_prediction[0].round(3))

### Multiple Regression erkunden

Leider ist multiple lineare Regression nicht so einfach darzustellen, aber die Verwendung von nur zwei unabhängigen Variablen zur Vorhersage einer abhängigen Variable kann immer noch grafisch mit einem 3D-Plot dargestellt werden.

Jetzt werden wir versuchen, die Datenpunkte und unser angepasstes lineares Modell auf den drei Achsen (`horsepower`, `weight` und `mpg`) darzustellen. Mit zwei Variablen anstelle einer unabhängigen Variable wird unser Modell nicht durch eine Linie dargestellt, sondern durch eine 2D-Ebene.

In der Zelle unten finden Sie den Code zum Plotten dieses 3D-Plots, Sie müssen jedoch zuerst einige Variablen definieren.

Der Befehl
```Python:
%matplotlib widget
```
ermöglicht interaktive 3D-Plots, sodass Sie Ihre Ansicht auf die Daten ändern können. (Wenn der Plot nach Ausführung der Zelle nicht angezeigt wird, führen Sie die Zelle einfach erneut aus... das sollte das Problem beheben :) )

In [ ]:
# We adapted this plot from https://www.datarobot.com/blog/multiple-regression-using-statsmodels/#appendix

from mpl_toolkits.mplot3d import Axes3D

# Enable interactive plot
%matplotlib widget

# You need to define some variables for the intercept and coefficients of your model
intercept = lin_reg2.intercept_
coef_horsepower, coef_weight = lin_reg2.coef_

# Create horsepower/weight grid for the 3D plot
xx1, xx2 = np.meshgrid(np.linspace(X2.horsepower.min(), X2.horsepower.max(), 100), 
                       np.linspace(X2.weight.min(), X2.weight.max(), 100))

# Plot the hyperplane (HP) by evaluating the parameters on the grid
# The following line is our regression equation 
Z = intercept + coef_horsepower * xx1 + coef_weight * xx2


# Create 3D figure in matplotlib
fig = plt.figure(figsize=(12, 8))
ax = Axes3D(fig, azim=-115, elev=15, auto_add_to_figure=False)
fig.add_axes(ax)

# Plot the hyperplane
surf = ax.plot_surface(xx1, xx2, Z, cmap=plt.cm.RdBu_r, alpha=0.8, linewidth=0)

# Calculate residuals
resid = y2 - y_hat2

# Plot data points - points over the HP are white, points below are black
ax.scatter(X2[resid >= 0].horsepower, X2[resid >= 0].weight, y2[resid >= 0], color='black', alpha=1.0, facecolor='white')
ax.scatter(X2[resid < 0].horsepower, X2[resid < 0].weight, y2[resid < 0], color='black', alpha=1.0)

# set axis labels
ax.set_xlabel('horsepower')
ax.set_ylabel('weight')
ax.set_zlabel('mpg');

Um die Interaktivität für die kommenden Plots zu deaktivieren, müssen wir `%matplotlib inline` erneut ausführen.

In [ ]:
%matplotlib inline

## Weitere Variablen hinzufügen

Versuchen wir nun, `mpg` mit `displacement`, `horsepower`, `weight` und `acceleration` vorherzusagen.

In [ ]:
# Print correlation of variables
cars[['displacement', 'horsepower', 'weight', 'acceleration']].corr()

Sie sehen, dass es ziemlich viele Korrelationen zwischen diesen Variablen gibt! Diese Korrelationen können auch in den Streudiagrammen gesehen werden:

In [ ]:
# Plot pair plot of potential features
sns.pairplot(cars[['displacement', 'horsepower', 'weight', 'acceleration']]);

In [ ]:
# Define and fit model with multiple variables
X3 = cars[['horsepower', 'weight', 'displacement', 'acceleration']]
y3 = cars.mpg

# Fit linear regression model
lin_reg3 = LinearRegression()
lin_reg3.fit(X3, y3)

# Calculate intercept and coefficient
intercept = lin_reg3.intercept_
coefficients = lin_reg3.coef_
print("Intercept:", intercept.round(4))
print("Coefficients:", coefficients.round(4))

In [ ]:
# Calculate r-squared 
y_hat3 = lin_reg3.predict(X3)
print("R-squared:", round(r2_score(y3, y_hat3),3))

Sie sollten Ihr Modell immer hinterfragen! Hier sind einige Fragen, die Sie jetzt beantworten können sollten:

1. Wie gut ist die Modellanpassung?

In [ ]:
"""
Das Modell passt deutlich besser als die einfachere Variante mit weniger Features.
Mit einem R-squared von rund 0.707 erklaert es etwa 70.7% der Varianz in mpg.
Das ist ordentlich, aber immer noch kein perfektes Modell, weil fast 29.3% unerklaert bleiben.
"""

2. Was ist die Regressionsgleichung?

In [ ]:
intercept = lin_reg3.intercept_
coef_horsepower, coef_weight, coef_displacement, coef_acceleration = lin_reg3.coef_

print(
    "mpg = "
    f"{intercept:.4f} "
    f"{coef_horsepower:+.4f} * horsepower "
    f"{coef_weight:+.4f} * weight "
    f"{coef_displacement:+.4f} * displacement "
    f"{coef_acceleration:+.4f} * acceleration"
)

<br>

<details><summary>
Hier klicken für die Lösung
</summary>
$$
\hat{mpg} = 45.2511 - 0.0436 \times horsepower - 0.0053 \times weight - 0.0060 \times displacement - 0.0231 \times acceleration
$$
</details>

3. Wie können wir das Modell interpretieren?

In [ ]:
"""
Wenn wir die anderen Variablen konstant halten, sinkt das erwartete mpg
- um etwa 0.0436 fuer jede zusaetzliche horsepower,
- um etwa 0.0053 fuer jede zusaetzliche Gewichtseinheit,
- um etwa 0.0060 fuer jede zusaetzliche displacement-Einheit und
- um etwa 0.0231 fuer jede zusaetzliche acceleration-Einheit.

Alle vier Koeffizienten sind negativ, also gehen groessere Werte dieser Features im Modell
mit einem niedrigeren Kraftstoffwirkungsgrad einher.
"""

## $R^2$ vs. Adjustiertes $R^2$

Wann immer wir mit multipler linearer Regression arbeiten, sollte $R^2$ nicht die Metrik Ihrer Wahl zur Bewertung der Modellanpassung sein. Durch das Hinzufügen neuer Features zu unserem Modell wird das $R^2$ entweder gleich bleiben oder sich erhöhen, selbst wenn diese zusätzlichen Features keine Beziehung zu unserer Zielvariablen haben. Dies führt zu einer überwältigenden Versuchung, immer mehr Features in unser Modell aufzunehmen. Das ist jedoch keine gute Idee! Normalerweise wollen wir ein gutes Modell trainieren, **aber** wir wollen es auf die einfachstmögliche Weise tun.

Hier kommt das **adjustierte $R^2$** zur Hilfe. Adjustiertes $R^2$ wird uns dafür bestrafen, mehr Features hinzuzufügen, die unser bestehendes Modell tatsächlich nicht verbessern. Für eine einfache lineare Regression sind $R^2$ und adjustiertes $R^2$ fast gleich. Je mehr nicht-signifikante Features wir einem Modell hinzufügen, desto größer wird die Lücke zwischen diesen beiden Metriken. Darüber hinaus ist das adjustierte $R^2$ eine nützliche Metrik, wenn wir mehrere Modelle mit unterschiedlichen Mengen an Features vergleichen möchten.

Die Formel des adjustierten $R^2$ lautet:
$$
R_a^2 = 1 - \frac{(1 - R^2) (n - 1)}{n - p - 1}
$$

wobei $n$ die Anzahl der Beobachtungen im Datensatz und $p$ die Anzahl der Features ist.

Leider bietet scikit-learn keine eingebaute Funktion zur Berechnung des adjustierten $R^2$. Aber wir können unsere eigene Funktion unter Verwendung der obigen Formel schreiben:

In [ ]:
# Define function for calculating adjusted r-squared
def adjusted_r_squared(r_squared, X):
    adjusted_r2 = 1 - ((1 - r_squared) * (len(X) - 1) / (len(X) - X.shape[1] - 1))
    return adjusted_r2 

### Übung

Berechnen Sie das adjustierte $R^2$ für die drei Modelle, die Sie in diesem Notebook trainiert haben. Welches ist das beste Modell laut dieser Metrik?

In [ ]:
adjusted_r2_results = pd.DataFrame(
    {
        "model": [
            "horsepower",
            "horsepower + weight",
            "horsepower + weight + displacement + acceleration",
        ],
        "r_squared": [
            r2_score(y, y_hat),
            r2_score(y2, y_hat2),
            r2_score(y3, y_hat3),
        ],
        "adjusted_r_squared": [
            adjusted_r_squared(r2_score(y, y_hat), X),
            adjusted_r_squared(r2_score(y2, y_hat2), X2),
            adjusted_r_squared(r2_score(y3, y_hat3), X3),
        ],
    }
)

adjusted_r2_results.sort_values("adjusted_r_squared", ascending=False)

## Zusammenfassung

- Wir erweitern das lineare Regressionsmodell, um viele erklärende Variablen/Features einzuschließen.
- Alle erklärenden Variablen sollten voneinander unabhängig sein.
- $R^2$ ermöglicht es uns zu messen, wie gut ein Modell zu den Daten passt.
- Adjustiertes $R^2$ bestraft das Hinzufügen nicht-nützlicher erklärender Variablen. Es ist nützlich für den Vergleich von Modellen mit unterschiedlichen Anzahlen unabhängiger Variablen.